# fMRI Reliability Pipeline - V2 (Parameter Grid)
- Analyzes `coordinatesAll_param.csv`: multi-echo (`me`) data rebuilt over a grid of preprocessing parameters.
- Grid (x07.blur_censor_param.csh): smoothing kernel `-FWHM` {0, 2, 4, 6} mm x motion censor limit {0.15, 0.3, 0.45}.
- Encoded in `runTag` as `param-b{FWHM}m{motion}` ('.' -> 'p'), e.g. `param-b4m0p3` -> FWHM 4 mm, motion 0.3.
- Only the `abs` peak sign is used, to match the coordinates in `coordinatesAll_original.csv` (main.ipynb).

In [ ]:
import os
import re
import pandas as pd
import itertools
import numpy as np

from scipy.spatial.distance import pdist
from scipy.stats import ttest_rel
from statsmodels.stats.anova import AnovaRM

# Plotting Packages
import seaborn as sns
import matplotlib.pyplot as plt
import ptitprince as pt # Only needed for raincloud plot
import configFunctions as config

# Variables
dataFile = 'coordinatesAll_param.csv' # Path to the CSV file containing the coordinates data
peakSign = 'abs' # 'abs' matches the original main.ipynb coordinates; 'pos'/'neg' also available

# Output directory
output_dir = os.path.join('output', 'param')
os.makedirs(output_dir, exist_ok=True)

### Required files
- 'coordinatesAll_param.csv' - generated by serverFunctions, final output of mergeOutputs.sh, run over the paramScript grid.

In [ ]:
# A configuration function to make output images easier to modify for publication.
config.svg_editing()

In [ ]:
# load data, inspect. Keep subject/session as strings so leading zeros survive.
df = pd.read_csv(dataFile, dtype={'subject': str, 'session': str})
print(f'{len(df)} rows, {df["subject"].nunique()} subjects')
print(df['sign'].value_counts().to_string())
df.head()

### Intermediate clean up
- Keep only the `abs` peak sign.
- Keep only the grid variants (`param-b{F}m{M}`). Dropped:
  - `param` - the base afni_proc run (FWHM 4, motion 0.3). `param-b4m0p3` repeats its settings, so it would double that grid cell.
  - Untagged rows (empty `runTag`) - older runs that predate the grid.
- Decode `fwhm` and `motion` from the run tag.

In [ ]:
df = df[df['sign'] == peakSign].copy()

# Decode 'param-b4m0p3' -> fwhm '4', motion '0.3'
runTagPattern = re.compile(r'-b(\d+)m(\d+p\d+)$')

def decode_run_tag(tag):
    m = runTagPattern.search(tag) if isinstance(tag, str) else None
    if m is None:
        return pd.Series({'fwhm': None, 'motion': None})
    return pd.Series({'fwhm': m.group(1), 'motion': m.group(2).replace('p', '.')})

df[['fwhm', 'motion']] = df['runTag'].apply(decode_run_tag)

print('Rows dropped (not a grid variant):')
print(df.loc[df['fwhm'].isna(), 'runTag'].fillna('<untagged>').value_counts().to_string())
df = df[df['fwhm'].notna()].copy()

print(f'\n{len(df)} rows remaining')
pd.crosstab(df['fwhm'], df['motion'])

In [ ]:
# Pull out and show the rows which include any nans (peakZ excluded - missing peaks also lack X/Y/Z)
rows_with_nan = df[df[['X', 'Y', 'Z']].isna().any(axis=1)]
rows_with_nan.to_csv(os.path.join(output_dir, 'coordinatesAll_param_NaNRows.csv'), index=False)
subj_with_nan = rows_with_nan['subject'].unique()
print(f'Subjects with NaNs - {len(subj_with_nan)}')
print(f'List: {list(subj_with_nan)}')
print(f'Dataframe filtered to remove subjects with any NaNs')

df = df[~df['subject'].isin(subj_with_nan)]
print(f'{len(df["subject"].unique())} Subjects Remaining...')

### Mean pairwise distance per subject x condition
- For each subject x FWHM x motion x sgACC mask x dlPFC mask, the mean Euclidean distance across sessions.

In [ ]:
# Group by subject and condition
grouping_vars = ['subject', 'fwhm', 'motion', 'target', 'clustMask']

combodf = (df.groupby(grouping_vars)
             .apply(lambda g: pd.Series({
                 'n_sessions': len(g),
                 'mean_distance': round(pdist(g[['X', 'Y', 'Z']], metric='euclidean').mean(), 2)
             }), include_groups=False)
             .reset_index())
combodf['n_sessions'] = combodf['n_sessions'].astype(int)

print('Sessions per cell:')
print(combodf['n_sessions'].value_counts().to_string())
combodf.head()

In [ ]:
# Repeated-measures ANOVA needs every subject in every cell; drop subjects with missing cells.
nCells = combodf[grouping_vars[1:]].drop_duplicates().shape[0]
cellsPerSubj = combodf.groupby('subject').size()
incomplete = cellsPerSubj[cellsPerSubj < nCells].index.tolist()
print(f'{nCells} cells per subject. Subjects with missing cells - {len(incomplete)}: {incomplete}')

combodf = combodf[~combodf['subject'].isin(incomplete)].reset_index(drop=True)
print(f'{combodf["subject"].nunique()} Subjects Remaining...')

In [ ]:
# Export all pairwise distances for each analysis variant to CSV
csv_out = True  # Toggle: save all pairwise distances to CSV

if csv_out:
    pairwise_rows = []
    df_kept = df[df['subject'].isin(combodf['subject'])]
    for keys, g in df_kept.groupby(grouping_vars):
        g = g.sort_values('session')
        coords = g[['X', 'Y', 'Z']].values
        sessions = g['session'].values
        dists = pdist(coords, metric='euclidean')
        for pair_idx, (i, j) in enumerate(itertools.combinations(range(len(coords)), 2)):
            pairwise_rows.append({**dict(zip(grouping_vars, keys)),
                                  'session_i': sessions[i],
                                  'session_j': sessions[j],
                                  'euclidean_distance': round(dists[pair_idx], 2)})

    pairwise_df = pd.DataFrame(pairwise_rows)
    pairwise_csv_path = os.path.join(output_dir, 'pairwise_distances_param.csv')
    pairwise_df.to_csv(pairwise_csv_path, index=False)
    print(f'Pairwise distances saved to {pairwise_csv_path} ({len(pairwise_df)} rows)')
    pairwise_df.head(10)

# Plotting

In [ ]:
targetName = 'sgACC' # For plotting, what does the 'target' mask represent?
clusterRegion = 'dlPFC' # for plotting, what does the 'clusterMask' mask represent?

# Factors of interest: smoothing kernel and motion censoring.
groupVars = ['fwhm', 'motion'] # Matching df column names
groupVarNames = ['Smoothing Kernel', 'Motion Censoring']

# Sequence of statistical tests and plotting
orderVec = [['0', '2', '4', '6'],
            ['0.15', '0.3', '0.45']]

# Display names
nameVec = [['None', '2 mm', '4 mm', '6 mm'],
           ['0.15 mm', '0.3 mm', '0.45 mm']]

for var, order in zip(groupVars, orderVec):
    combodf[var] = pd.Categorical(combodf[var], categories=order, ordered=True)

In [ ]:
def plot_significance_brackets(ax, comparisons, y_min, y_max):
    """
    Plot significance brackets and stars on an axis.

    Parameters:
    -----------
    ax : matplotlib axis
        The axis to plot on
    comparisons : list of dict
        List of comparisons, each containing 'pos1', 'pos2', and 'pval'
    y_min : float
        Minimum y-value in the data
    y_max : float
        Maximum y-value in the data

    Returns:
    --------
    tuple : (new_y_min, new_y_max) adjusted limits to accommodate brackets
    """
    y_range = y_max - y_min

    if not comparisons:
        return (y_min - 0.05 * y_range, y_max + 0.05 * y_range)

    bracket_height_start = y_max + 0.02 * y_range
    bracket_spacing = 0.08 * y_range

    for comp_idx, comp in enumerate(comparisons):
        x1 = comp['pos1']
        x2 = comp['pos2']
        pval = comp['pval']

        # Determine star based on p-value
        if pval < 0.001:
            star = '***'
        elif pval < 0.01:
            star = '**'
        elif pval < 0.05:
            star = '*'
        else:
            star = 'ns'

        # Calculate bracket height (stack them)
        bracket_y = bracket_height_start + (comp_idx * bracket_spacing)
        star_y = bracket_y + 0.002 * y_range

        # Draw horizontal line
        ax.plot([x1, x2], [bracket_y, bracket_y], 'k-', linewidth=1.5)

        # Draw vertical ticks on both ends
        tick_height = 0.015 * y_range
        ax.plot([x1, x1], [bracket_y - tick_height, bracket_y], 'k-', linewidth=1.5)
        ax.plot([x2, x2], [bracket_y - tick_height, bracket_y], 'k-', linewidth=1.5)

        # Add star
        ax.text((x1 + x2) / 2, star_y, star, ha='center', va='bottom',
                fontsize=14, fontweight='bold')

    # Return adjusted y-limits to accommodate all brackets
    max_bracket_y = bracket_height_start + (len(comparisons) - 1) * bracket_spacing
    return (y_min - 0.05 * y_range, max_bracket_y + 0.10 * y_range)

### Statistical tests
- One-way RM ANOVA per factor, aggregating across the other factors so each subject has one value per level.
- Post-hoc paired t-tests, Bonferroni corrected.

In [ ]:
# Storage for statistical results
stat_results = {}

for idx, var in enumerate(groupVars):
    # Aggregate across other within-subject factors so each subject
    # has one value per level of the current factor
    agg_df = combodf.groupby(['subject', var], observed=True)['mean_distance'].mean().reset_index()
    agg_df = agg_df.sort_values('subject')

    groups = orderVec[idx]
    group_data = [agg_df[agg_df[var] == val]['mean_distance'].values for val in groups]

    # Repeated measures ANOVA (both factors have more than 2 levels)
    res = AnovaRM(data=agg_df, depvar='mean_distance', subject='subject', within=[var]).fit()
    stat = res.anova_table['F Value'].values[0]
    p_val = res.anova_table['Pr > F'].values[0]

    print(f'{groupVarNames[idx]}: among {groups} Repeated Measures ANOVA')
    print(res)

    # Post-hoc: pairwise paired t-tests with Bonferroni correction
    n_comparisons = len(groups) * (len(groups) - 1) // 2

    print(f'{groupVarNames[idx]} - Post-hoc Paired t-tests (Bonferroni corrected):')
    print('=' * 70)

    pairwise_results = []
    for g1, g2 in itertools.combinations(range(len(groups)), 2):
        t_stat, t_pval = ttest_rel(group_data[g1], group_data[g2])
        corrected_pval = min(t_pval * n_comparisons, 1.0)
        result = {
            'group1': groups[g1],
            'group2': groups[g2],
            'meandiff': group_data[g2].mean() - group_data[g1].mean(),
            't_stat': t_stat,
            'p_raw': t_pval,
            'p_corrected': corrected_pval,
            'reject': corrected_pval < 0.05
        }
        pairwise_results.append(result)
        sig = '*' if result['reject'] else ''
        print(f"  {result['group1']:>8} vs {result['group2']:<8}: "
              f"diff={result['meandiff']:>7.3f}, t={result['t_stat']:>7.3f}, "
              f"p_raw={result['p_raw']:.2e}, p_corr={result['p_corrected']:.2e} {sig}")
    print()

    # Extract significant comparisons for plotting, shortest brackets first
    group_positions = {group: i for i, group in enumerate(groups)}
    comparisons = [{'group1': r['group1'], 'group2': r['group2'], 'pval': r['p_corrected'],
                    'pos1': group_positions[r['group1']], 'pos2': group_positions[r['group2']]}
                   for r in pairwise_results if r['reject']]
    comparisons.sort(key=lambda x: abs(x['pos2'] - x['pos1']))

    stat_results[var] = {
        'test': 'anova',
        'stat': stat,
        'p_val': p_val,
        'stat_name': 'F',
        'groups': groups,
        'agg_df': agg_df,
        'pairwise': pairwise_results,
        'comparisons': comparisons
    }

print('\n' + '='*60)
print('Statistical analysis complete. Results stored for plotting.')
print('='*60)

### Raincloud plot - Smoothing kernel and motion censoring
- Each point is one subject's mean pairwise distance, averaged across the other factors (the same values tested above).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for idx, (var, varPlot) in enumerate(zip(groupVars, groupVarNames)):
    plot_df = stat_results[var]['agg_df']

    pt.RainCloud(data=plot_df, x=var, y='mean_distance', ax=axes[idx],
        palette='Set2', hue=var, bw=0.3, width_viol=0.7, width_box=0.3,
        orient='v', alpha=0.6, dodge=False, pointplot=False, point_size=3,
        linewidth=2, jitter=0.15, box_showfliers=False, box_linewidth=2)
    if axes[idx].get_legend() is not None:
        axes[idx].get_legend().remove()

    axes[idx].set_xticks(range(len(nameVec[idx])))
    axes[idx].set_xticklabels(nameVec[idx])

    # Open up the plot to ensure the peaks for each distribution are visible.
    percSpread = 0.4 # Defines how much you want to extend past current limits.
    xmin, xmax = axes[idx].get_xlim()
    axes[idx].set_xlim((xmin - percSpread, xmax + percSpread))

    # Get statistical results from the analysis cell
    results = stat_results[var]

    y_max = plot_df['mean_distance'].max()
    y_min = plot_df['mean_distance'].min()
    new_y_min, new_y_max = plot_significance_brackets(
        ax=axes[idx], comparisons=results['comparisons'], y_min=y_min, y_max=y_max
    )
    axes[idx].set_ylim(new_y_min, new_y_max)

    # Add title with statistics (two lines, different sizes)
    axes[idx].set_title(f'{varPlot}', fontsize=14, fontweight='bold', pad=22)
    axes[idx].text(0.5, 1.01, f'{results["stat_name"]}={results["stat"]:.2f}, p={results["p_val"]:.2e}',
                   transform=axes[idx].transAxes, ha='center', va='bottom', fontsize=12)
    axes[idx].set_xlabel('', fontsize=11)
    axes[idx].set_ylabel('Mean Pairwise Distance (mm)', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'Rainplot_param.svg'))
plt.show()

# 2-Way Repeated Measures ANOVA (FWHM x motion)
- Tests both main effects and their interaction, averaging over the sgACC and dlPFC masks.
- As in main.ipynb, the main effects match the one-way analyses above (each effect has its own error term).

In [ ]:
agg2_df = combodf.groupby(['subject', 'fwhm', 'motion'], observed=True)['mean_distance'].mean().reset_index()

res_2way = AnovaRM(data=agg2_df, depvar='mean_distance', subject='subject',
                   within=['fwhm', 'motion']).fit()
print('2-Way Repeated Measures ANOVA (fwhm x motion)')
print('=' * 60)
print(res_2way)
anova_2way_table = res_2way.anova_table

### Raincloud plot - every FWHM x motion cell
- Motion censoring nested within each smoothing kernel, to show the interaction.

In [ ]:
motionPalette = dict(zip(orderVec[1], sns.color_palette('Set2', len(orderVec[1]))))

fig, axes = plt.subplots(1, len(orderVec[0]), figsize=(16, 5), sharey=True)

for idx, (fw, fwName) in enumerate(zip(orderVec[0], nameVec[0])):
    cell_df = agg2_df[agg2_df['fwhm'] == fw]

    pt.RainCloud(data=cell_df, x='motion', y='mean_distance', ax=axes[idx],
        palette=motionPalette, hue='motion', bw=0.3, width_viol=0.7, width_box=0.3,
        orient='v', alpha=0.6, dodge=False, pointplot=False, point_size=3,
        linewidth=2, jitter=0.15, box_showfliers=False, box_linewidth=2)
    if axes[idx].get_legend() is not None:
        axes[idx].get_legend().remove()

    axes[idx].set_xticks(range(len(nameVec[1])))
    axes[idx].set_xticklabels(nameVec[1])
    xmin, xmax = axes[idx].get_xlim()
    axes[idx].set_xlim((xmin - 0.4, xmax + 0.4))

    axes[idx].set_title(f'FWHM: {fwName}', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Motion Censoring', fontsize=11)
    axes[idx].set_ylabel('Mean Pairwise Distance (mm)' if idx == 0 else '', fontsize=11)

f_int = anova_2way_table.loc['fwhm:motion', 'F Value']
p_int = anova_2way_table.loc['fwhm:motion', 'Pr > F']
plt.suptitle(f'Smoothing x Motion Censoring - interaction F={f_int:.2f}, p={p_int:.2e}',
             fontsize=16, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'Rainplot_param_interaction.svg'))
plt.show()

# Full 4-Way Repeated Measures ANOVA
- fwhm x motion x target x clustMask, using every cell of `combodf` (72 per subject).

In [ ]:
res_full = AnovaRM(data=combodf, depvar='mean_distance', subject='subject',
                   within=['fwhm', 'motion', 'target', 'clustMask']).fit()
print('4-Way Repeated Measures ANOVA')
print('=' * 60)
print(res_full)
full_anova_table = res_full.anova_table

### Export statistics to file

In [ ]:
# Save all statistical results to a text file
stats_path = os.path.join(output_dir, 'stats_param.txt')

with open(stats_path, 'w') as f:
    f.write('Statistical Results Summary - Parameter Grid\n')
    f.write(f'Data: {dataFile}, peak sign: {peakSign}\n')
    f.write(f'N subjects: {combodf["subject"].nunique()}\n')
    f.write('=' * 90 + '\n\n')

    # --- Separate one-way RM ANOVAs ---
    f.write('SEPARATE ONE-WAY ANALYSES\n')
    f.write('=' * 90 + '\n\n')

    for idx, var in enumerate(groupVars):
        results = stat_results[var]
        groups = results['groups']
        agg_df = results['agg_df']
        f.write(f'Variable: {groupVarNames[idx]} ({var})\n')
        f.write(f'Groups: {groups}\n')
        f.write('Test: Repeated Measures ANOVA\n')
        f.write(f'{results["stat_name"]} = {results["stat"]:.4f}, p = {results["p_val"]:.2e}\n')

        # Group descriptives
        f.write('\nGroup descriptives:\n')
        for g in groups:
            vals = agg_df[agg_df[var] == g]['mean_distance'].values
            f.write(f'  {g:<12}: mean={vals.mean():.3f}, SD={vals.std(ddof=1):.3f}, n={len(vals)}\n')

        # Pairwise comparisons
        f.write(f'\nPost-hoc Paired t-tests (Bonferroni corrected, k={len(results["pairwise"])}):\n')
        f.write(f'  {"Comparison":<22} {"mean_diff":>10} {"t":>10} {"p_raw":>12} {"p_corr":>12} {"sig":>5}\n')
        f.write('  ' + '-' * 75 + '\n')
        for r in results['pairwise']:
            sig = '*' if r['reject'] else ''
            f.write(f'  {r["group1"]:>8} vs {r["group2"]:<8}   {r["meandiff"]:>10.3f} {r["t_stat"]:>10.3f} '
                    f'{r["p_raw"]:>12.2e} {r["p_corrected"]:>12.2e} {sig:>5}\n')

        f.write('\n' + '=' * 90 + '\n\n')

    # --- Factorial RM ANOVAs ---
    for title, table in [('2-WAY REPEATED MEASURES ANOVA (fwhm x motion)', anova_2way_table),
                         ('4-WAY REPEATED MEASURES ANOVA (fwhm x motion x target x clustMask)', full_anova_table)]:
        f.write(title + '\n')
        f.write('=' * 90 + '\n\n')
        f.write(table.to_string())
        f.write('\n\n')

        sig_effects = table[table['Pr > F'] < 0.05]
        if len(sig_effects) > 0:
            f.write('Significant effects (p < 0.05):\n')
            for effect_name, row in sig_effects.iterrows():
                f.write(f'  {effect_name}: F={row["F Value"]:.4f}, p={row["Pr > F"]:.2e}\n')
        else:
            f.write('No significant effects at p < 0.05.\n')
        f.write('\n' + '=' * 90 + '\n\n')

print(f'Statistics saved to {stats_path}')